# ClarioAI System Test Notebook

Notebook ini menguji seluruh komponen sistem ClarioAI secara terstruktur:

| # | Seksi | Deskripsi |
|---|-------|-----------|
| 1 | Imports & Environment | Verifikasi semua package tersedia |
| 2 | State Schema | Dataclass dan EBPState |
| 3 | Utility Functions | `extract_json`, `format_constraints` |
| 4 | LLM Connectivity | Koneksi ke DeepInfra / Qwen |
| 5 | Internet Search Tool | BrightData API |
| 6 | MongoDB | Koneksi, save, load state |
| 7 | Agent Nodes | Market Scout, Strategic Architect, Financial Analyst, Ethics Agent, Orchestrator |
| 8 | Graph Pipeline | Build graph & jalankan 1 iterasi penuh |
| 9 | End-to-End | Full run dengan business scenario nyata |

---
## 1. Imports & Environment Check

In [1]:
import sys, importlib

REQUIRED_PACKAGES = [
    "langchain",
    "langchain_openai",
    "langchain_core",
    "langgraph",
    "pymongo",
    "requests",
]

missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg}")
        missing.append(pkg)

if missing:
    print(f"\nInstall missing: pip install {' '.join(missing)}")
else:
    print("\nSemua package tersedia.")

  [OK] langchain
  [OK] langchain_openai
  [OK] langchain_core
  [OK] langgraph
  [OK] pymongo
  [OK] requests

Semua package tersedia.


In [2]:
# Pastikan root project ada di sys.path agar import relatif berjalan
import os
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: d:\OneDrive\Kuliah\Semester 6\Capstone\ClarioAI


---
## 2. State Schema Tests

In [3]:
from states.schema import (
    BussinessConstraints,
    MarketScoutReport,
    StrategicReport,
    FinancialAnalysisReport,
    EthicsAnalysisReport,
    EBPState,
)
print("Import states.schema: OK")

Import states.schema: OK


In [4]:
# --- Test BussinessConstraints (intentional typo sesuai codebase) ---
constraints = BussinessConstraints(
    sector_and_domain="EdTech / Platform Belajar Online",
    audience="Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)",
    initial_prompt="Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional",
)
print("BussinessConstraints:")
print(f"  sector_and_domain : {constraints.sector_and_domain}")
print(f"  audience          : {constraints.audience}")
print(f"  initial_prompt    : {constraints.initial_prompt}")

BussinessConstraints:
  sector_and_domain : EdTech / Platform Belajar Online
  audience          : Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)
  initial_prompt    : Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional


In [5]:
# --- Test report dataclasses ---
market_report = MarketScoutReport(
    ideas=["Ide A", "Ide B"],
    agent_explanation="Penjelasan dari market scout.",
)
strategic_report = StrategicReport(
    swot_analysis="SWOT placeholder",
    pastel_analysis="PESTEL placeholder",
)
financial_report = FinancialAnalysisReport(analysis_result="Proyeksi keuangan placeholder")
ethics_report    = EthicsAnalysisReport(analysis_result="Analisis etika placeholder")

print("MarketScoutReport  :", market_report)
print("StrategicReport    :", strategic_report)
print("FinancialAnalysis  :", financial_report)
print("EthicsAnalysis     :", ethics_report)

MarketScoutReport  : MarketScoutReport(ideas=['Ide A', 'Ide B'], agent_explanation='Penjelasan dari market scout.')
StrategicReport    : StrategicReport(swot_analysis='SWOT placeholder', pastel_analysis='PESTEL placeholder')
FinancialAnalysis  : FinancialAnalysisReport(analysis_result='Proyeksi keuangan placeholder')
EthicsAnalysis     : EthicsAnalysisReport(analysis_result='Analisis etika placeholder')


In [6]:
# --- Test EBPState instantiation ---
import uuid

sample_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "user_id": "test_user_notebook",
    "bussiness_constraints": constraints,
    "market_scout_report": None,
    "strategic_report": None,
    "financial_analysis_report": None,
    "ethics_analysis_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "messages": [],
    "iteration": 0,
    "max_iterations": 3,
    "user_feedback": None,
}
print("EBPState state_id :", sample_state["state_id"])
print("approval_status   :", sample_state["approval_status"])
print("iteration         :", sample_state["iteration"], "/", sample_state["max_iterations"])
print("market_report None:", sample_state["market_scout_report"] is None)

EBPState state_id : 885e87ea-fbe1-4814-b3c6-d073e61ff709
approval_status   : pending
iteration         : 0 / 3
market_report None: True


---
## 3. Utility Functions Tests

In [7]:
from functions.agent_utils import extract_json, format_constraints
print("Import functions.agent_utils: OK")

Import functions.agent_utils: OK


In [8]:
# --- extract_json: markdown fenced ---
fenced = '''
Berikut hasilnya:
```json
{"status": "approved", "score": 8.5, "reason": "Market viable"}
```
'''
result = extract_json(fenced)
assert result["status"] == "approved", "Fenced JSON parse gagal"
assert result["score"] == 8.5
print("extract_json (fenced)  :", result)

extract_json (fenced)  : {'status': 'approved', 'score': 8.5, 'reason': 'Market viable'}


In [9]:
# --- extract_json: bare JSON ---
bare = 'Analisis selesai. {"ideas": ["A", "B"], "confidence": 0.9}'
result2 = extract_json(bare)
assert result2["ideas"] == ["A", "B"]
print("extract_json (bare)    :", result2)

extract_json (bare)    : {'ideas': ['A', 'B'], 'confidence': 0.9}


In [10]:
# --- extract_json: fallback ke {"raw": ...} ---
plain = "Tidak ada JSON di sini sama sekali."
result3 = extract_json(plain)
assert "raw" in result3
print("extract_json (fallback):", result3)

extract_json (fallback): {'raw': 'Tidak ada JSON di sini sama sekali.'}


In [11]:
# --- format_constraints ---
formatted = format_constraints(constraints)
print("format_constraints output:")
print(formatted)
assert "EdTech" in formatted
assert "Pelajar SMA" in formatted

format_constraints output:
Sector/Domain: EdTech / Platform Belajar Online
Target Audience: Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)
Business Idea: Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional


In [12]:
# --- format_constraints dengan None ---
none_fmt = format_constraints(None)
assert "No constraints" in none_fmt
print("format_constraints(None):", none_fmt)

format_constraints(None): No constraints provided.


---
## 4. LLM Connectivity Test

> Sel ini melakukan satu panggilan ke DeepInfra API. Perlu koneksi internet.

In [13]:
from functions.llm import get_llm, MODEL_NAME, DEEPINFRA_BASE_URL
print(f"Model      : {MODEL_NAME}")
print(f"Base URL   : {DEEPINFRA_BASE_URL}")

Model      : Qwen/Qwen3.5-397B-A17B
Base URL   : https://api.deepinfra.com/v1/openai


In [14]:
llm = get_llm(temperature=0.3)

from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="Balas hanya dengan kata: PING")])
print("LLM response:", response.content[:200])

LLM response: 

PING


---
## 5. Internet Search Tool Test

> Memerlukan koneksi internet dan BrightData API key yang valid.

In [15]:
from tools.internet_search import internet_search
print("Import tools.internet_search: OK")
print("Tool name :", internet_search.name)
print("Tool desc :", internet_search.description[:80], "...")

Import tools.internet_search: OK
Tool name : internet_search
Tool desc : Search the internet for real-time business and market information.
    Use this  ...


In [16]:
query = "Ide bisnis kuliner"
print(f"Searching: '{query}' ...\n")

search_result = internet_search.invoke({"query": query})
# print(search_result[:1000])  # Tampilkan 1000 char pertama

Searching: 'Ide bisnis kuliner' ...



In [19]:
print(search_result)

Search results for 'Ide bisnis kuliner':


1. 10 Ide Bisnis Kuliner Kekinian, Bisa Dicoba untuk Pemula
   https://ocean.bca.co.id/artikel/10-ide-bisnis-kuliner

2. 30 Ide Jualan Makanan Kekinian Untuk Bisnis Kuliner
   https://www.unileverfoodsolutions.co.id/id/inspirasi-chef/ukm-sukses/10-ide-jualan-makanan-kekinian-untuk-bisnis-kuliner.html

3. 23 Ide Bisnis Kuliner Modal Kecil Tapi Cuan Besar!
   https://www.lalamove.com/id/blog/bisnis-kuliner-modal-kecil/

4. Lagi Tren Matcha? Coba 5 Ide Bisnis Kuliner Menarik ...
   https://www.ciputramakassar.ac.id/man/lagi-tren-matcha-coba-5-ide-bisnis-kuliner-menarik-lainnya/

5. 10 Cara Memulai Usaha Kuliner Plus Rekomendasi 7 Ide ...
   https://glints.com/id/lowongan/cara-memulai-usaha-kuliner/


-- Page Content --

[https://ocean.bca.co.id/artikel/10-ide-bisnis-kuliner]
10 Ide Bisnis Kuliner Kekinian, Bisa Dicoba untuk Pemula | Ocean by BCA Dengan mengakses situs ini, Anda telah menyetujui penggunaan cookies dari kami. ID Masuk • Rekomendasi

In [18]:
stop

NameError: name 'stop' is not defined

---
## 6. MongoDB Tests

> Memerlukan MongoDB berjalan di `localhost:27017`.

In [19]:
from functions.mongodb import create_new_state, save_state, load_state, MONGO_URI, DB_NAME
print(f"MongoDB URI : {MONGO_URI}")
print(f"Database    : {DB_NAME}")

MongoDB URI : mongodb://localhost:27017
Database    : clario_ai


In [20]:
# --- Test koneksi MongoDB ---
from pymongo import MongoClient
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    client.admin.command("ping")
    print("MongoDB connection: OK")
    MONGO_AVAILABLE = True
except Exception as e:
    print(f"MongoDB connection: GAGAL ({e})")
    print("Test seksi ini di-skip. Jalankan MongoDB terlebih dahulu.")
    MONGO_AVAILABLE = False

MongoDB connection: OK


In [21]:
if MONGO_AVAILABLE:
    # --- create_new_state ---
    new_state = create_new_state(
        constraints=constraints,
        user_id="notebook_test_user",
        max_iterations=3,
    )
    print("State dibuat, ID:", new_state["state_id"])
    print("approval_status :", new_state["approval_status"])
    print("iteration       :", new_state["iteration"])
    TEST_STATE_ID = new_state["state_id"]
else:
    TEST_STATE_ID = None
    print("Skip — MongoDB tidak tersedia.")

State dibuat, ID: 71b6577e-c2e9-403e-b4aa-58d479220f02
approval_status : pending
iteration       : 0


In [22]:
if MONGO_AVAILABLE:
    # --- save_state ---
    new_state["market_scout_report"] = market_report
    saved_id = save_state(new_state)
    print("save_state OK, returned ID:", saved_id)
else:
    print("Skip.")

save_state OK, returned ID: 71b6577e-c2e9-403e-b4aa-58d479220f02


In [23]:
if MONGO_AVAILABLE:
    # --- load_state ---
    loaded = load_state(TEST_STATE_ID)
    assert loaded is not None, "State tidak ditemukan setelah save!"
    assert loaded["state_id"] == TEST_STATE_ID
    assert loaded["market_scout_report"] is not None
    assert loaded["market_scout_report"].ideas == ["Ide A", "Ide B"]
    print("load_state OK")
    print("  state_id        :", loaded["state_id"])
    print("  user_id         :", loaded["user_id"])
    print("  market ideas    :", loaded["market_scout_report"].ideas)

    # --- load non-existent ---
    not_found = load_state("00000000-0000-0000-0000-000000000000")
    assert not_found is None
    print("  load non-existent: None (benar)")
else:
    print("Skip.")

load_state OK
  state_id        : 71b6577e-c2e9-403e-b4aa-58d479220f02
  user_id         : notebook_test_user
  market ideas    : ['Ide A', 'Ide B']
  load non-existent: None (benar)


In [24]:
if MONGO_AVAILABLE:
    # --- Serialisasi messages (HumanMessage, AIMessage, SystemMessage) ---
    from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
    from functions.mongodb import _serialize_messages, _deserialize_messages

    msgs = [
        SystemMessage(content="System prompt"),
        HumanMessage(content="Halo"),
        AIMessage(content="Halo juga!"),
    ]
    serialized = _serialize_messages(msgs)
    deserialized = _deserialize_messages(serialized)

    assert len(deserialized) == 3
    assert deserialized[1].content == "Halo"
    print("Message serialization/deserialization: OK")
    print("  Types:", [type(m).__name__ for m in deserialized])
else:
    print("Skip.")

Message serialization/deserialization: OK
  Types: ['SystemMessage', 'HumanMessage', 'AIMessage']


---
## 7. Agent Node Tests

Setiap agent diuji secara **terpisah** dengan state minimal. Ini menguji logika prompt + parsing output tiap node tanpa menjalankan graph penuh.

> Sel-sel ini memanggil LLM secara nyata — estimasi 30-120 detik per agent.

In [25]:
# Helper: buat state bersih untuk setiap test
def make_fresh_state(iteration=0, with_market=False, with_strategic=False, with_financial=False, with_ethics=False):
    state: EBPState = {
        "state_id": str(uuid.uuid4()),
        "user_id": "notebook_agent_test",
        "bussiness_constraints": constraints,
        "market_scout_report": market_report if with_market else None,
        "strategic_report": strategic_report if with_strategic else None,
        "financial_analysis_report": financial_report if with_financial else None,
        "ethics_analysis_report": ethics_report if with_ethics else None,
        "approval_status": "pending",
        "orchestrator_feedback": None,
        "messages": [],
        "iteration": iteration,
        "max_iterations": 3,
        "user_feedback": None,
    }
    return state

print("Helper make_fresh_state: siap")

Helper make_fresh_state: siap


### 7.1 Market Scout Agent

In [26]:
from nodes.market_scout import market_scout_node

state_ms = make_fresh_state(iteration=0)
print("Menjalankan Market Scout Agent...")
result_ms = market_scout_node(state_ms)

report_ms = result_ms.get("market_scout_report")
assert report_ms is not None, "market_scout_report seharusnya tidak None!"
assert isinstance(report_ms.ideas, list), "ideas harus berupa list"
assert len(report_ms.ideas) > 0, "ideas tidak boleh kosong"

print("\n[Market Scout] LULUS")
print(f"  Jumlah ide       : {len(report_ms.ideas)}")
for i, idea in enumerate(report_ms.ideas, 1):
    print(f"  Ide {i}: {idea[:100]}")
print(f"\n  Penjelasan (150 char): {report_ms.agent_explanation[:150]}...")

Menjalankan Market Scout Agent...

[Market Scout] LULUS
  Jumlah ide       : 3
  Ide 1: Micro-learning SNBT Prep Platform with Short-Form Video Content: Launch a TikTok-style learning plat
  Ide 2: Professional Certification Micro-Course Hub for University Students: Create a platform offering 2-5 
  Ide 3: AI-Powered Adaptive Learning Engine for Personalized Test Prep: Develop an AI-driven micro-learning 

  Penjelasan (150 char): The Indonesian EdTech market presents significant opportunities for micro-learning platforms targeting students aged 16-22. Market research reveals th...


### 7.2 Strategic Architect Agent

In [27]:
from nodes.strategic_architect import strategic_architect_node

# Strategic Architect butuh market_scout_report
state_sa = make_fresh_state(iteration=0, with_market=True)
# Gunakan hasil nyata dari market scout jika tersedia
if report_ms is not None:
    state_sa["market_scout_report"] = report_ms

print("Menjalankan Strategic Architect Agent...")
result_sa = strategic_architect_node(state_sa)

report_sa = result_sa.get("strategic_report")
assert report_sa is not None, "strategic_report seharusnya tidak None!"
assert report_sa.swot_analysis, "swot_analysis tidak boleh kosong"
assert report_sa.pastel_analysis, "pastel_analysis tidak boleh kosong"

print("\n[Strategic Architect] LULUS")
print(f"  SWOT (200 char)  : {report_sa.swot_analysis[:200]}...")
print(f"  PESTEL (200 char): {report_sa.pastel_analysis[:200]}...")

Menjalankan Strategic Architect Agent...

[Strategic Architect] LULUS
  SWOT (200 char)  : ## SWOT Analysis

**Strengths:**
- Micro-learning format aligns with Gen Z preferences (TikTok has 100M+ Indonesian users, proving short-form content adoption)
- Price point (IDR 49,000-99,000/month) ...
  PESTEL (200 char): ## PESTEL Analysis

**Political:**
Indonesia's government actively supports digital education transformation under the "Golden Indonesia 2045" vision. The Ministry of Education, Culture, Research and ...


### 7.3 Financial Analyst Agent

In [42]:
from nodes.financial_analyst import financial_analyst_node

state_fa = make_fresh_state(iteration=0, with_market=True, with_strategic=True)
# state_fa['market_scout_report'] = None  
state_fa["market_scout_report"] = report_ms
if report_sa is not None:
    state_fa["strategic_report"] = report_sa

print("Menjalankan Financial Analyst Agent...")
result_fa = financial_analyst_node(state_fa)

report_fa = result_fa.get("financial_analysis_report")
assert report_fa is not None, "financial_analysis_report seharusnya tidak None!"
assert report_fa.analysis_result, "analysis_result tidak boleh kosong"

print("\n[Financial Analyst] LULUS")
print(f"  Hasil (300 char) : {report_fa.analysis_result[:300]}...")

Menjalankan Financial Analyst Agent...

[Financial Analyst] LULUS
  Hasil (300 char) : # Financial Analysis: Micro-Learning EdTech Platform Indonesia

## Executive Summary
This analysis evaluates the financial viability of a micro-learning platform targeting SNBT exam preparation (785,058 annual test-takers, 29.4% pass rate) and professional certification courses for Indonesian univer...


In [44]:
print(f"  Hasil (300 char) : {report_fa.analysis_result}...")

  Hasil (300 char) : # Financial Analysis: Micro-Learning EdTech Platform Indonesia

## Executive Summary
This analysis evaluates the financial viability of a micro-learning platform targeting SNBT exam preparation (785,058 annual test-takers, 29.4% pass rate) and professional certification courses for Indonesian university students (8.7M). The platform leverages short-form video content (60-90 seconds) at IDR 49,000-99,000/month, undercutting competitors like Ruangguru (IDR 399,000-2,550,000/packages).

---

## 1. Initial Investment Estimate

### Startup Costs Breakdown (Year 0)

| Category | Item | Cost (IDR) | Cost (USD) |
|----------|------|------------|------------|
| **Legal & Setup** | PT PMA establishment, OSS registration, NIB licensing | 45,000,000 | $2,900 |
| **Technology** | Platform development (MVP), cloud infrastructure, AI engine | 350,000,000 | $22,500 |
| **Content Production** | Initial video library (500 micro-lessons @ 500K/video) | 250,000,000 | $16,000 |
| **Off

In [41]:
report_ms.ideas

["Micro-learning SNBT Prep Platform with Short-Form Video Content: Launch a TikTok-style learning platform delivering 60-90 second video lessons specifically for SNBT exam preparation. With 785,058 students taking SNBT 2024 and only 29.4% passing rate, there's massive demand for effective test prep. Unlike Ruangguru's longer video format, this platform would leverage Gen Z's preference for short-form content (proven by TikTok's 100M+ Indonesian users), offering bite-sized lessons on UTBK topics like quantitative reasoning, literacy, and subject-specific content. Pricing at IDR 49,000-99,000/month would undercut competitors while targeting the price-sensitive student segment.",
 "Professional Certification Micro-Course Hub for University Students: Create a platform offering 2-5 hour micro-courses for industry-recognized certifications (Google, Microsoft, AWS, digital marketing, data analytics) aligned with Indonesia's Kartu Prakerja (Pre-Employment Card) program that provides up to IDR 

In [38]:
print(report_fa.analysis_result)

# Financial Analysis: Micro-Learning EdTech Platform Indonesia

## Executive Summary

This analysis evaluates the financial viability of a micro-learning video platform targeting Indonesian high school students (ages 16-22) for SNBT exam preparation and professional certification. Based on current market data from Indonesia's EdTech sector, this venture shows promising unit economics with a clear path to profitability within 18-24 months.

---

## 1. Initial Investment Estimate

### Startup Costs Breakdown (Year 0)

| Category | Item | Cost (IDR) | Cost (USD) |
|----------|------|------------|------------|
| **Technology Development** | Platform development (MVP) | 800,000,000 | $50,000 |
| | Mobile app (iOS/Android) | 400,000,000 | $25,000 |
| | AI adaptive learning system | 300,000,000 | $18,750 |
| | Cloud infrastructure setup | 100,000,000 | $6,250 |
| **Content Production** | Video content creation (500 hours) | 500,000,000 | $31,250 |
| | Professional certification modules | 200,

In [ ]:
print(report_fa.analysis_result)

# Financial Analysis: Micro-Learning EdTech Platform Indonesia

## Executive Summary
This financial analysis evaluates a micro-learning EdTech platform targeting Indonesian high school students (SNBT preparation) and university students (professional certification). The analysis incorporates real market data from Indonesia's EdTech sector, including pricing benchmarks from Ruangguru, Zenius, and Pahamify, operational cost structures, and funding landscape insights.

---

## 1. Initial Investment Estimate

### Startup Costs Breakdown (Year 0)

| Category | Item | Estimated Cost (IDR) | Estimated Cost (USD)* |
|----------|------|---------------------|----------------------|
| **Technology Development** | Platform development (MVP) | Rp 800.000.000 | $50,000 |
| | Mobile app (iOS & Android) | Rp 500.000.000 | $31,250 |
| | Cloud infrastructure setup | Rp 150.000.000 | $9,375 |
| | Adaptive learning algorithm | Rp 300.000.000 | $18,750 |
| **Content Production** | Video production (200 mic

### 7.4 Ethics Guardian Agent

In [45]:
from nodes.ethics_agent import ethics_agent_node

state_ea = make_fresh_state(iteration=0, with_market=True, with_strategic=True, with_financial=True)
state_ea["market_scout_report"] = report_ms
if report_sa: state_ea["strategic_report"] = report_sa
if report_fa: state_ea["financial_analysis_report"] = report_fa

print("Menjalankan Ethics Guardian Agent...")
result_ea = ethics_agent_node(state_ea)

report_ea = result_ea.get("ethics_analysis_report")
assert report_ea is not None, "ethics_analysis_report seharusnya tidak None!"
assert report_ea.analysis_result, "analysis_result tidak boleh kosong"

print("\n[Ethics Guardian] LULUS")
print(f"  Hasil (300 char) : {report_ea.analysis_result[:300]}...")

Menjalankan Ethics Guardian Agent...

[Ethics Guardian] LULUS
  Hasil (300 char) : # Ethics & Compliance Analysis: EdTech Micro-Learning Platform Indonesia

## Executive Summary
This analysis evaluates the legal compliance and ethical considerations for a micro-learning EdTech platform targeting Indonesian students (ages 16-22) for SNBT exam preparation and professional certificat...


### 7.5 Lead Orchestrator — iterasi pertama (belum ada laporan)

In [46]:
from nodes.lead_orchestrator import lead_orchestrator_node

# Iterasi 0: semua report None → orchestrator harus route ke market scout
state_orch_init = make_fresh_state(iteration=0)
print("Menjalankan Lead Orchestrator (iterasi awal, tanpa laporan)...")
result_orch_init = lead_orchestrator_node(state_orch_init)

print("\n[Orchestrator iterasi awal] Output:")
print(f"  approval_status      : {result_orch_init.get('approval_status')}")
print(f"  orchestrator_feedback: {str(result_orch_init.get('orchestrator_feedback', ''))[:150]}")
print(f"  iteration            : {result_orch_init.get('iteration')}")

Menjalankan Lead Orchestrator (iterasi awal, tanpa laporan)...

[Orchestrator iterasi awal] Output:
  approval_status      : pending
  orchestrator_feedback: None
  iteration            : None


In [47]:
# Iterasi 1: semua report sudah ada → orchestrator evaluasi dan beri verdict
state_orch_eval = make_fresh_state(iteration=1, with_market=True, with_strategic=True, with_financial=True, with_ethics=True)
state_orch_eval["market_scout_report"] = report_ms
if report_sa: state_orch_eval["strategic_report"] = report_sa
if report_fa: state_orch_eval["financial_analysis_report"] = report_fa
if report_ea: state_orch_eval["ethics_analysis_report"] = report_ea

print("Menjalankan Lead Orchestrator (evaluasi laporan lengkap)...")
result_orch_eval = lead_orchestrator_node(state_orch_eval)

status = result_orch_eval.get("approval_status")
assert status in ("approved", "rejected", "pending"), f"Status tidak valid: {status}"

print("\n[Orchestrator evaluasi] LULUS")
print(f"  approval_status      : {status}")
print(f"  iteration            : {result_orch_eval.get('iteration')}")
feedback = result_orch_eval.get('orchestrator_feedback') or ""
print(f"  feedback (200 char)  : {feedback[:200]}")

Menjalankan Lead Orchestrator (evaluasi laporan lengkap)...

[Orchestrator evaluasi] LULUS
  approval_status      : rejected
  iteration            : 2
  feedback (200 char)  : Market Scout: Justify 0.5% Year 1 penetration assumption with competitor benchmarking or pilot data. Strategy: Add implementation roadmap that integrates Ethics compliance timeline into go-to-market s


---
## 8. Graph Pipeline Test

Uji build graph LangGraph dan jalankan **satu iterasi penuh** (satu putaran pipeline semua agent).

In [48]:
from graphs.ebp_graph import build_graph, _route_from_orchestrator
print("Import graphs.ebp_graph: OK")

Import graphs.ebp_graph: OK


In [49]:
# --- Test routing function secara unit ---

# Case 1: approved → END
route_approved = _route_from_orchestrator({"approval_status": "approved", "iteration": 1, "max_iterations": 3})
assert route_approved == "__end__", f"Expected __end__, got {route_approved}"
print("Route approved     :", route_approved)

# Case 2: max_iterations tercapai → END
route_maxed = _route_from_orchestrator({"approval_status": "rejected", "iteration": 3, "max_iterations": 3})
assert route_maxed == "__end__"
print("Route max_iter hit :", route_maxed)

# Case 3: rejected, masih ada iterasi → kembali ke market_scout
route_retry = _route_from_orchestrator({"approval_status": "rejected", "iteration": 1, "max_iterations": 3})
assert route_retry == "market_scout"
print("Route retry        :", route_retry)

print("\nSemua routing case: LULUS")

Route approved     : __end__
Route max_iter hit : __end__
Route retry        : market_scout

Semua routing case: LULUS


In [50]:
# --- Build graph ---
graph = build_graph()
print("Graph berhasil dikompilasi:", type(graph).__name__)
print("Nodes:", list(graph.nodes.keys()) if hasattr(graph, 'nodes') else "(gunakan graph.get_graph() untuk detail)")

Graph berhasil dikompilasi: CompiledStateGraph
Nodes: ['__start__', 'lead_orchestrator', 'market_scout', 'strategic_architect', 'financial_analyst', 'ethics_agent']


---
## 9. End-to-End Integration Test

Jalankan graph secara penuh dengan **max_iterations=1** agar hanya 1 siklus pipeline yang berjalan. Ini mensimulasikan alur produksi nyata.

> **Perhatian:** Sel ini memanggil semua 5 agent dan internet search. Estimasi waktu: **5-15 menit**.

In [51]:
import time

e2e_constraints = BussinessConstraints(
    sector_and_domain="EdTech / Platform Belajar Online",
    audience="Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)",
    initial_prompt="Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional",
)

initial_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "user_id": "e2e_test",
    "bussiness_constraints": e2e_constraints,
    "market_scout_report": None,
    "strategic_report": None,
    "financial_analysis_report": None,
    "ethics_analysis_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "messages": [],
    "iteration": 0,
    "max_iterations": 1,  # 1 iterasi saja untuk test cepat
    "user_feedback": None,
}

print(f"State ID: {initial_state['state_id']}")
print("Menjalankan graph end-to-end (max_iterations=1)...")
print("=" * 60)

start = time.time()
final_state = graph.invoke(initial_state)
elapsed = time.time() - start

print(f"\nGraph selesai dalam {elapsed:.1f} detik")

State ID: d90c716e-9fa3-41e6-be78-e0b795b2f764
Menjalankan graph end-to-end (max_iterations=1)...

Graph selesai dalam 1002.2 detik


In [52]:
# --- Validasi output end-to-end ---
print("=" * 60)
print("HASIL END-TO-END")
print("=" * 60)

print(f"approval_status     : {final_state.get('approval_status')}")
print(f"iteration           : {final_state.get('iteration')}")
print(f"messages count      : {len(final_state.get('messages', []))}")

ms = final_state.get("market_scout_report")
sa = final_state.get("strategic_report")
fa = final_state.get("financial_analysis_report")
ea = final_state.get("ethics_analysis_report")

print(f"\nmarket_scout_report      : {'Ada' if ms else 'TIDAK ADA'}")
print(f"strategic_report         : {'Ada' if sa else 'TIDAK ADA'}")
print(f"financial_analysis_report: {'Ada' if fa else 'TIDAK ADA'}")
print(f"ethics_analysis_report   : {'Ada' if ea else 'TIDAK ADA'}")

HASIL END-TO-END
approval_status     : rejected
iteration           : 1
messages count      : 30

market_scout_report      : Ada
strategic_report         : Ada
financial_analysis_report: Ada
ethics_analysis_report   : Ada


In [53]:
# --- Tampilkan ringkasan tiap laporan ---
DIVIDER = "-" * 60

if ms:
    print(DIVIDER)
    print("MARKET SCOUT")
    print(DIVIDER)
    for i, idea in enumerate(ms.ideas, 1):
        print(f"  Ide {i}: {idea}")
    print(f"\n  Penjelasan: {ms.agent_explanation[:400]}")

if sa:
    print(DIVIDER)
    print("STRATEGIC ARCHITECT")
    print(DIVIDER)
    print(f"  SWOT  : {sa.swot_analysis[:400]}")
    print(f"  PESTEL: {sa.pastel_analysis[:400]}")

if fa:
    print(DIVIDER)
    print("FINANCIAL ANALYST")
    print(DIVIDER)
    print(f"  {fa.analysis_result[:500]}")

if ea:
    print(DIVIDER)
    print("ETHICS GUARDIAN")
    print(DIVIDER)
    print(f"  {ea.analysis_result[:500]}")

feedback = final_state.get("orchestrator_feedback") or ""
if feedback:
    print(DIVIDER)
    print("ORCHESTRATOR FEEDBACK")
    print(DIVIDER)
    print(f"  {feedback[:400]}")

------------------------------------------------------------
MARKET SCOUT
------------------------------------------------------------
  Ide 1: Platform Micro-Learning Video Pendek untuk Persiapan SNBT dengan Format TikTok-Style: Mengingat 77% penetrasi internet Indonesia dan preferensi Gen Z terhadap konten video pendek, platform ini menawarkan materi belajar SNBT dalam format 3-5 menit per topik dengan fitur gamifikasi, tryout adaptif, dan komunitas belajar. Pasar EdTech Indonesia diproyeksikan tumbuh dari USD 1,423.1 Juta (2025) menjadi USD 9,356.6 Juta (2034) dengan CAGR 23.28%, dengan segmen ujian masuk universitas menjadi katalis utama.
  Ide 2: Platform Sertifikasi Profesional Mikro untuk Mahasiswa dengan Partnership Industri: Menyasar demand sertifikasi profesional yang meningkat (3 juta+ sertifikasi telah diterbitkan di Indonesia), platform ini menyediakan kursus mikro terakreditasi untuk skill digital (data analytics, digital marketing, AI) dengan durasi 10-15 menit per modul

In [54]:
# --- Assertions akhir end-to-end ---
assert final_state.get("approval_status") in ("approved", "rejected", "pending"), \
    "approval_status harus valid"
assert final_state.get("market_scout_report") is not None, \
    "Market scout report seharusnya terisi"
assert final_state.get("strategic_report") is not None, \
    "Strategic report seharusnya terisi"
assert final_state.get("financial_analysis_report") is not None, \
    "Financial report seharusnya terisi"
assert final_state.get("ethics_analysis_report") is not None, \
    "Ethics report seharusnya terisi"

print("SEMUA ASSERTION END-TO-END: LULUS")
print(f"\nSistem ClarioAI berjalan normal. Status akhir: {final_state['approval_status'].upper()}")

SEMUA ASSERTION END-TO-END: LULUS

Sistem ClarioAI berjalan normal. Status akhir: REJECTED
